RAG with LangChain

In [1]:
%pip install langchain langchain_community langchain-chroma chromadb sentence-transformers dotenv pypdf langchain-huggingface

  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.2 kB)
  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached pypdf-6.7.5-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.2.17-py3-none-any.whl.metadata (4.4 kB)
  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm6

In [2]:
#loading the documents
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("air-rag.pdf")
docs = loader.load()

docs[0].metadata

/opt/homebrew/Caskroom/miniforge/base/envs/maclean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 178 0 (offset 0)
Ignoring wrong pointing object 346 0 (offset 0)
Ignoring wrong pointing object 347 0 (offset 0)


{'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext',
 'creator': 'Elsevier',
 'creationdate': "D:20260131174331Z00'00'",
 'title': 'Adaptive iterative retrieval for enhanced retrieval-augmented generation',
 'author': 'Wenhan Han',
 'subject': 'Neurocomputing, 666 (2026) 132272. doi:10.1016/j.neucom.2025.132272',
 'moddate': "D:20260131174331Z00'00'",
 'keywords': 'LLM,RAG',
 'aapl:keywords': "['LLM,RAG']",
 'source': 'air-rag.pdf',
 'total_pages': 14,
 'page': 0,
 'page_label': '1'}

In [3]:
#split into chunks
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(docs)
print("Length of chunks : ",len(chunks))

Length of chunks :  104


In [4]:
#load embedding model form hugging face
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise ValueError("Missing Hugging Face token. Set HUGGINGFACE_TOKEN in your .env file.")

login(token = hf_token)


In [5]:
#creating embeddings
from langchain_huggingface import HuggingFaceEmbeddings

loading_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = loading_model.embed_documents([c.page_content for c in chunks])
len(embeddings), len(embeddings[0])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11610.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(104, 384)

In [ ]:
#storing in vector db
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = chunks,
    collection_name = "docs_collection",
    embedding = loading_model,
    persist_directory = "./chroma_db"
)

In [ ]:
#retrieving from vector db
retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k' : 3}
)

results = retriever.invoke("What is the title of the paper?")
print(results)

[Document(id='c1810674-cd8b-4d85-b9ff-7926ac3afb6a', metadata={'title': 'Adaptive iterative retrieval for enhanced retrieval-augmented generation', 'subject': 'Neurocomputing, 666 (2026) 132272. doi:10.1016/j.neucom.2025.132272', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'total_pages': 14, 'moddate': "D:20260131174331Z00'00'", 'source': 'air-rag.pdf', 'page_label': '11', 'creator': 'Elsevier', 'keywords': 'LLM,RAG', 'aapl:keywords': "['LLM,RAG']", 'author': 'Wenhan Han', 'page': 10, 'creationdate': "D:20260131174331Z00'00'"}, page_content='Iter-2 Sentences\ns_1\x01 Kate Millett Katherine Murray Millett (September 14, 1934 – September 6, 2017) was an\nAmerican feminist writer, educator, artist, and activist.\x01\n \ns_2 She attended Oxford University and was the first American woman to be awarded a degree\nwith first-class honors after studying at St Hilda’s College, Oxford.\x01\n \ns_3 She has been described as “a seminal influence on second-wave feminism” and i